# Establish concordance between acts, editions and inventories

# Environment

## install

In [38]:
!pip install lxml

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------- ----------- 2.9/4.0 MB 15.4 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 14.4 MB/s eta 0:00:00


## import

In [3]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import re

# Data

## list of editions and inventories

In [116]:
# Define directories
editions_dir = Path('../Editions')
inventories_dir = Path('../Inventories')

# Get .xml files in ../Editions (non-recursively)
editions_xml = [str(f) for f in editions_dir.glob('*.xml')]

# Get .xml files in ../Inventories (recursively)
inventories_xml = [str(f) for f in inventories_dir.rglob('*.xml')]

# Combine lists
all_xml_files = editions_xml + inventories_xml

# Print or use the list
print(all_xml_files)


['..\\Editions\\Guerin_tome1-tome12.xml', '..\\Editions\\Longnon.xml', '..\\Editions\\morchesne.xml', '..\\Editions\\Viard.xml', '..\\Inventories\\Geographic\\Paris_AN_JJ_inventaire_Gascogne.xml', '..\\Inventories\\Geographic\\Paris_AN_JJ_inventaire_Languedoc.xml', '..\\Inventories\\Geographic\\Paris_AN_JJ_inventaire_Loire.xml', '..\\Inventories\\Geographic\\Paris_AN_JJ_inventaire_Rouergue.xml', '..\\Inventories\\Systematic\\Paris_AN_JJ_inventaire_IR421-JJA-JJ79A.xml', '..\\Inventories\\Systematic\\Paris_AN_JJ_inventaire_IR422-JJ80-155.xml', '..\\Inventories\\Systematic\\Paris_AN_JJ_inventaire_IR423-JJ156-211.xml', '..\\Inventories\\Systematic\\Paris_AN_JJ_inventaire_JJ37-50.xml', '..\\Inventories\\Systematic\\Paris_AN_JJ_inventaire_JJ37-50_index.xml', '..\\Inventories\\Systematic\\Paris_AN_JJ_inventaire_JJ50-60.xml', '..\\Inventories\\Systematic\\Paris_AN_JJ_inventaire_JJ61-64.xml', '..\\Inventories\\Systematic\\Paris_AN_JJ_inventaire_JJ65A-JJ69.xml', '..\\Inventories\\Systematic\\Par

## List of acts

In [118]:

# Replace with your actual path if different
acts_path = 'Acts-JJ96-JJ195_complete.csv'

# Read CSV into DataFrame
acts = pd.read_csv(acts_path, sep=";")

# Display the first few rows
print(acts.shape)
print(acts.head())


(39989, 3)
  Register Act_number Folio Number
0     JJ96          1            6
1     JJ96          2            7
2     JJ96          3           10
3     JJ96          4      12v-15v
4     JJ96          5        Vacat


# Concordance

## Merging

### Sorting

In [123]:
%%time

# Define custom sort key
# Custom sort key: split into (leading_number, suffix_string)
def natural_sort_key(val):
    # Match: optional letters + digits + optional suffix
    match = re.match(r'^([A-Za-z]*)(\d+)(.*)$', str(val))
    if match:
        prefix = match.group(1)
        number = int(match.group(2))
        suffix = match.group(3)
        return (prefix, number, suffix)
    else:
        return ('', float('inf'), val)  # Push non-matching to end



# Sort the DataFrame
acts_sorted = acts.sort_values(
    by=["Register", "Act_number"],
    key=lambda col: col.map(natural_sort_key)
).reset_index(drop=True)

acts_sorted.to_csv('Acts-JJ96-JJ195_sorted.csv', sep=';', index=False)
acts_sorted.head()

CPU times: total: 484 ms
Wall time: 482 ms


,Register,Act_number,Folio Number
0,JJ96,1,6
1,JJ96,2,7
2,JJ96,3,10
3,JJ96,4,12v-15v
4,JJ96,5,Vacat


### Merging proper

In [127]:
%%time

import pandas as pd
from lxml import etree
import os

# Fonction de normalisation
def normalize(val):
    return str(val).replace(" ", "").replace("/", "").replace(".", "").strip() if val else ""

# Ajouter les clés normalisées au DataFrame source
acts_sorted['Register_clean'] = acts_sorted['Register'].map(normalize)
acts_sorted['Act_number_clean'] = acts_sorted['Act_number'].map(normalize)

# Répertoire de sortie pour les CSV
output_dir = "witness_csvs"
os.makedirs(output_dir, exist_ok=True)

# Merge progressif
acts_enriched = acts_sorted.copy()

# Déclaration du namespace TEI
NSMAP = {'tei': 'http://www.tei-c.org/ns/1.0'}

for xml_path in all_xml_files:
    file = os.path.basename(xml_path)
    print(file)
    tree = etree.parse(xml_path)
    witnesses = tree.xpath('//tei:witness', namespaces=NSMAP)

    records = []

    for wit in witnesses:
        reg_elem = wit.xpath('./tei:msDesc/tei:msIdentifier/tei:idno', namespaces=NSMAP)
        act_elem = wit.xpath('./tei:idno', namespaces=NSMAP)

        if len(reg_elem) == 0 or len(act_elem) == 0:
            continue

        reg_val = normalize(reg_elem[0].text)
        act_val = normalize(act_elem[0].text)

        # Head depuis ancestor::front[1]/head
        head_nodes = wit.xpath('ancestor::tei:front[1]/tei:head', namespaces=NSMAP)
        if len(head_nodes) > 0 and head_nodes[0].text:
            head_text = head_nodes[0].text.strip()
        else:
            head_text = ""
            print(f"[head not found] Register: {reg_val}, Act: {act_val}")

        # Locus
        locus_nodes = wit.xpath('tei:locus', namespaces=NSMAP)
        if len(locus_nodes) > 0 and locus_nodes[0].text:
            locus_text = locus_nodes[0].text.strip()
        else:
            locus_text = ""
            
        records.append({
            f'{file}_Register': reg_val,
            f'{file}_Act_number': act_val,
            f'{file}_Head': normalize(head_text),
            f'{file}_Locus': normalize(locus_text),
            f'{file}_File': file
        })

    # Créer DataFrame pour ce fichier
    df = pd.DataFrame(records)

    if not df.empty:
        base_name = os.path.splitext(file)[0]
        csv_path = os.path.join(output_dir, f"witness_{base_name}.csv")
        df.to_csv(csv_path, sep=';', index=False)

        # Merge dans acts_enriched
        acts_enriched = pd.merge(
            acts_enriched,
            df,
            left_on=['Register_clean', 'Act_number_clean'],
            right_on=[f'{file}_Register', f'{file}_Act_number'],
            how='left',
            suffixes=('', f'_{base_name}')
        )



Guerin_tome1-tome12.xml
Longnon.xml
morchesne.xml
Viard.xml
Paris_AN_JJ_inventaire_Gascogne.xml
Paris_AN_JJ_inventaire_Languedoc.xml
[head not found] Register: JJ35, Act: 55
[head not found] Register: JJ36, Act: 53
Paris_AN_JJ_inventaire_Loire.xml
Paris_AN_JJ_inventaire_Rouergue.xml
Paris_AN_JJ_inventaire_IR421-JJA-JJ79A.xml
[head not found] Register: , Act: 
[head not found] Register: , Act: 
[head not found] Register: , Act: 
[head not found] Register: JJ1, Act: 
[head not found] Register: JJ1, Act: 
[head not found] Register: JJ1, Act: 
[head not found] Register: JJ1, Act: 
[head not found] Register: JJ1, Act: 
[head not found] Register: JJ1, Act: 
[head not found] Register: JJ1, Act: 
[head not found] Register: JJ1, Act: 
[head not found] Register: JJ1, Act: 
[head not found] Register: JJ1, Act: 
[head not found] Register: JJ1, Act: 
[head not found] Register: JJ1, Act: 
[head not found] Register: JJ1, Act: 
[head not found] Register: JJ1, Act: 
[head not found] Register: JJ1, Act:

In [129]:
acts_enriched.to_csv("Acts-JJ96-JJ195_enriched.csv", sep=";")

### Locus

In [131]:
import numpy as np

# 1. Trouver les colonnes contenant "Locus" dans leur nom
locus_cols = [col for col in acts_enriched.columns if 'Locus' in col]

# 2. Fonction pour concaténer les locus d'une ligne
def concat_locus_if_folio_empty(row):
    folio = row.get('Folio Number')
    if pd.isna(folio) or str(folio).strip() == '':
        values = [str(row[col]).strip() for col in locus_cols if pd.notna(row[col]) and str(row[col]).strip() != '']
        return '|'.join(values) if values else np.nan
    return folio  # garder la valeur existante sinon

# 3. Appliquer la fonction
acts_enriched['Folio Number'] = acts_enriched.apply(concat_locus_if_folio_empty, axis=1)

acts_enriched.to_csv("Acts-JJ96-JJ195_enriched_Locus.csv", sep=";")


## Manual checking and cleaning - not reproducible

## Checking consistency and exhaustiveness

### List of acts

In [102]:
# Replace with your actual path if different
acts_path = 'Acts-JJ96-JJ195_enriched_Locus.csv'

# Read CSV into DataFrame
acts = pd.read_csv(acts_path, sep=";", encoding='cp1252')
print(acts.shape)
acts.head()

(39832, 39)


C:\Users\stutzmann\AppData\Local\Temp\ipykernel_15936\1883618303.py:5: DtypeWarning: Columns (5,9,11,13,20,21,26,31) have mixed types. Specify dtype option on import or set low_memory=False.
  acts = pd.read_csv(acts_path, sep=";", encoding='cp1252')


,ID,Register,Act_number,Folio Number,Guerin_tome1-tome12.xml_Register,Guerin_tome1-tome12.xml_Act_number,Guerin_tome1-tome12.xml_Head,Guerin_tome1-tome12.xml_Locus,Guerin_tome1-tome12.xml_File,Longnon.xml_Register,...,Paris_AN_JJ_inventaire_Loire.xml_Register,Paris_AN_JJ_inventaire_Loire.xml_Act_number,Paris_AN_JJ_inventaire_Loire.xml_Head,Paris_AN_JJ_inventaire_Loire.xml_Locus,Paris_AN_JJ_inventaire_Loire.xml_File,Paris_AN_JJ_inventaire_Rouergue.xml_Register,Paris_AN_JJ_inventaire_Rouergue.xml_Act_number,Paris_AN_JJ_inventaire_Rouergue.xml_Head,Paris_AN_JJ_inventaire_Rouergue.xml_Locus,Paris_AN_JJ_inventaire_Rouergue.xml_File
0,0,JJ96,1,6,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,JJ96,2,7,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,JJ96,3,10,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,JJ96,4,12v-15v,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,JJ96,5,Vacat,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### bis, ter etc. from inventories and editions

In [ ]:
import os
import pandas as pd

# Répertoire contenant les fichiers witness_*.csv
folder_path = "./witness_csvs"
bis_ter_quater_entries = []

# Mots-clés à rechercher
suffixes = ['bis', 'ter', 'quater']

for filename in os.listdir(folder_path):
    if filename.startswith("witness_") and filename.endswith(".csv"):
        specific_filename = filename[len("witness_"):-len(".csv")]
        file_path = os.path.join(folder_path, filename)
        print(filename)
        try:
            df = pd.read_csv(file_path, sep=';')
            
            # Nom des colonnes à extraire
            reg_col = f"{specific_filename}.xml_Register"
            act_col = f"{specific_filename}.xml_Act_number"
            head_col = f"{specific_filename}.xml_Head"

            # Filtrer les lignes avec 'bis', 'ter', 'quater' dans l'act number
            filtered = df[df[act_col].astype(str).str.contains(r'(?:bis|ter|quater)', case=False, na=False)]

            for _, row in filtered.iterrows():
                bis_ter_quater_entries.append([
                    specific_filename,
                    row.get(reg_col, ''),
                    row.get(act_col, ''),
                    row.get(head_col, '')
                ])
        
        except Exception as e:
            print(f"Erreur lors du traitement de {filename}: {e}")

# Résultat dans bis_ter_quater_entries
print(len(bis_ter_quater_entries))

In [96]:
# Entêtes pour le DataFrame
columns = ['File', 'Register', 'Act_number', 'Head']

# Créer un DataFrame à partir de la liste
df_out = pd.DataFrame(bis_ter_quater_entries, columns=columns)
print(df_out.shape)
# Sauvegarder en CSV
df_out.to_csv("acts_with_bis_ter_quater.csv", sep=';', index=False, encoding='utf-8-sig')

(608, 4)


In [114]:
# Normalisation des colonnes pour comparaison
def normalize(val):
    return str(val).replace(" ", "").replace("/", "").replace(".", "").strip() if val else ""


# Normaliser les couples dans acts
acts['Register_clean'] = acts['Register'].map(normalize)
acts['Act_number_clean'] = acts['Act_number'].map(normalize)

# Créer un set des couples existants pour recherche rapide
acts_keys = set(zip(acts['Register_clean'], acts['Act_number_clean']))

# Liste des entrées manquantes
missing_entries = []

# Parcours de df_out
for _, row in df_out.iterrows():
    reg = normalize(row['Register'])
    act = normalize(row['Act_number'])
    
    if (reg, act) not in acts_keys:
        missing_entries.append([
            row['File'],
            normalize(row['Register']),
            normalize(row['Act_number']),
            row['Head']
        ])

print(len(missing_entries))
missing_entries_df = pd.DataFrame(missing_entries, columns=columns) 
missing_entries_df.to_csv("missing_entries.csv", sep=";", index=False, encoding='utf-8-sig') 

507
